# SHIPP — Stage 11 Agent READ Validation

**Purpose:** validate the existing Agent READ implementation end-to-end without rebuilding it.

Validated flow:

```text
Real Request
    ↓
GoldReader.get_candidate_matches()
    ↓
Gold candidate listing_id
    ↓
ListingSearch.search()
    ↓
AI Search context restricted to Gold candidates
    ↓
LakebaseRepo.get_listing()
    ↓
Current operational listing state
    ↓
ShippAgent.chat()
    ↓
Grounded recommendation
```

## Important boundaries

- This notebook **must not call `save_item()` or `confirm_save()`**.
- The existing Agent source files are reused as-is.
- `ToolBox.dispatch()` and `ShippAgent.chat()` may append **audit rows** to `shipp.agent_activity`; that is expected.
- The notebook verifies that **no `saved_items` business write** is produced by Stage 11.
- Run the cells **top to bottom**.
- If any gate fails, stop at that gate and fix only the smallest proven cause.

**Target:** `Stage 11 — Agent READ = VALIDATED`

In [0]:
%uv pip install "psycopg[binary]>=3.2,<4" "openai>=1,<2"

In [0]:
%run ../common/shipp_silver_lib

## Gate 11.0 — Reuse the existing Agent implementation

This gate imports the current repository implementation.  
It does **not** create `retrieval.py` and does **not** modify Agent source files.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.0
# VERIFY EXISTING AGENT READ IMPLEMENTATION
# ============================================================

import os
import sys
import json
from datetime import datetime, timezone

agent_src = os.path.join(
    REPO_ROOT,
    "agent",
    "agent_ship",
    "src",
)

if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

print("Agent source:", agent_src)

from databricks.sdk import WorkspaceClient

from shipp.agent.agent import ShippAgent
from shipp.agent.config import AgentSettings
from shipp.agent.contracts import ActionStatus
from shipp.agent.gold import GoldReader
from shipp.agent.lakebase import LakebaseRepo
from shipp.agent.llm import DatabricksChatModel
from shipp.agent.search import ListingSearch
from shipp.agent.tools import ToolBox, ToolContext

assert callable(getattr(GoldReader, "get_candidate_matches", None))
assert callable(getattr(ListingSearch, "search", None))
assert callable(getattr(LakebaseRepo, "get_listing", None))
assert callable(getattr(ShippAgent, "chat", None))

print()
print("Existing Agent READ components:")
print("  GoldReader.get_candidate_matches : OK")
print("  ListingSearch.search             : OK")
print("  LakebaseRepo.get_listing         : OK")
print("  ToolBox                          : OK")
print("  ShippAgent.chat                  : OK")
print()
print("PASS Gate 11.0 — existing Agent READ implementation imported successfully")

## Gate 11.1 — Real Databricks resources

This gate discovers the SQL warehouses and serving endpoints visible to the current user.

The notebook creates two widgets at the top:

- `stage11_sql_warehouse_id`
- `stage11_llm_endpoint`

A usable SQL warehouse is selected automatically when possible.

For the LLM endpoint, choose the endpoint that SHIPP should use for the Agent.  
No secret is stored in this notebook.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.1A
# DISCOVER DATABRICKS RESOURCES
# READ ONLY
# ============================================================

import os
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

me = w.current_user.me().user_name
print("Databricks user:", me)

# ------------------------------------------------------------
# SQL WAREHOUSES
# ------------------------------------------------------------

warehouses = list(w.warehouses.list())

warehouse_rows = [
    (
        getattr(wh, "id", None),
        getattr(wh, "name", None),
        str(getattr(wh, "state", None)),
    )
    for wh in warehouses
]

print("\nVisible SQL warehouses:")

for row in warehouse_rows:
    print(row)

assert warehouse_rows, (
    "BLOCKED: no SQL warehouses are visible to this user."
)


def _is_running(state_text):
    return "RUNNING" in (state_text or "").upper()


running_ids = [
    row[0]
    for row in warehouse_rows
    if row[0] and _is_running(row[2])
]

all_warehouse_ids = [
    row[0]
    for row in warehouse_rows
    if row[0]
]

assert all_warehouse_ids, (
    "BLOCKED: no usable SQL warehouse IDs found."
)

default_warehouse_id = (
    os.getenv("SHIPP_SQL_WAREHOUSE_ID")
    or (
        running_ids[0]
        if running_ids
        else all_warehouse_ids[0]
    )
)

# ------------------------------------------------------------
# SERVING ENDPOINTS
# ------------------------------------------------------------

try:
    endpoint_objects = list(
        w.serving_endpoints.list()
    )

except Exception as exc:
    print(
        "\nWARNING: serving endpoint listing failed:",
        type(exc).__name__,
        str(exc),
    )

    endpoint_objects = []


endpoint_names = [
    getattr(ep, "name", None)
    for ep in endpoint_objects
    if getattr(ep, "name", None)
]

print("\nVisible serving endpoints:")

for name in endpoint_names:
    print(" ", name)

assert endpoint_names, (
    "BLOCKED: no serving endpoints are visible."
)

# ------------------------------------------------------------
# SELECT CORRECT SHIPP LLM ENDPOINT
# ------------------------------------------------------------

PREFERRED_LLM_ENDPOINT = "databricks-gpt-5-6-sol"

env_llm = os.getenv(
    "SHIPP_LLM_ENDPOINT",
    "",
).strip()

# IMPORTANT:
# Prefer the known Chat Completions compatible endpoint first.
if PREFERRED_LLM_ENDPOINT in endpoint_names:
    default_llm = PREFERRED_LLM_ENDPOINT

elif env_llm and env_llm in endpoint_names:
    default_llm = env_llm

else:
    default_llm = endpoint_names[0]

print(
    "\nDefault LLM endpoint chosen:",
    default_llm,
)

# ------------------------------------------------------------
# CREATE / REFRESH WIDGETS
# ------------------------------------------------------------

for widget_name in [
    "stage11_sql_warehouse_id",
    "stage11_llm_endpoint",
]:
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass


dbutils.widgets.dropdown(
    "stage11_sql_warehouse_id",
    default_warehouse_id,
    all_warehouse_ids,
    "Stage 11 SQL warehouse ID",
)


dbutils.widgets.dropdown(
    "stage11_llm_endpoint",
    default_llm,
    endpoint_names,
    "Stage 11 LLM endpoint",
)

# ------------------------------------------------------------
# READ THE NEW WIDGET VALUES
# ------------------------------------------------------------

SQL_WAREHOUSE_ID = (
    dbutils.widgets
    .get("stage11_sql_warehouse_id")
    .strip()
)

LLM_ENDPOINT = (
    dbutils.widgets
    .get("stage11_llm_endpoint")
    .strip()
)

print(
    "\nSelected SQL warehouse ID:",
    SQL_WAREHOUSE_ID,
)

print(
    "Selected LLM endpoint     :",
    LLM_ENDPOINT,
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert SQL_WAREHOUSE_ID, (
    "BLOCKED: SQL warehouse ID is empty."
)

assert LLM_ENDPOINT, (
    "BLOCKED: LLM endpoint is empty."
)

assert LLM_ENDPOINT == PREFERRED_LLM_ENDPOINT, (
    f"BLOCKED: wrong LLM endpoint selected: {LLM_ENDPOINT}. "
    f"Expected: {PREFERRED_LLM_ENDPOINT}"
)

print()
print(
    "PASS Gate 11.1A — Databricks resources discovered"
)

In [0]:
# ============================================================
# STAGE 11 — GATE 11.1A
# DISCOVER DATABRICKS RESOURCES
# READ ONLY
# ============================================================

w = WorkspaceClient()

me = w.current_user.me().user_name
print("Databricks user:", me)

# ------------------------------------------------------------
# SQL WAREHOUSES
# ------------------------------------------------------------

warehouses = list(w.warehouses.list())
warehouse_rows = []

for wh in warehouses:
    warehouse_rows.append(
        (
            getattr(wh, "id", None),
            getattr(wh, "name", None),
            str(getattr(wh, "state", None)),
        )
    )

print("\nVisible SQL warehouses:")
for row in warehouse_rows:
    print(row)

assert warehouse_rows, (
    "BLOCKED: no SQL warehouses are visible to this user."
)


def _is_running(state_text):
    return "RUNNING" in (state_text or "").upper()


running_ids = [
    row[0]
    for row in warehouse_rows
    if row[0] and _is_running(row[2])
]

all_warehouse_ids = [
    row[0]
    for row in warehouse_rows
    if row[0]
]

default_warehouse_id = (
    os.getenv("SHIPP_SQL_WAREHOUSE_ID")
    or (
        running_ids[0]
        if running_ids
        else all_warehouse_ids[0]
    )
)

# ------------------------------------------------------------
# SERVING ENDPOINTS
# ------------------------------------------------------------

try:
    endpoint_objects = list(
        w.serving_endpoints.list()
    )
except Exception as exc:
    print(
        "\nWARNING: serving endpoint listing failed:",
        type(exc).__name__,
        str(exc),
    )
    endpoint_objects = []


endpoint_names = [
    getattr(ep, "name", None)
    for ep in endpoint_objects
    if getattr(ep, "name", None)
]

print("\nVisible serving endpoints:")
for name in endpoint_names:
    print(" ", name)

# ------------------------------------------------------------
# SELECT CORRECT SHIPP LLM ENDPOINT
# ------------------------------------------------------------

env_llm = os.getenv(
    "SHIPP_LLM_ENDPOINT",
    "",
).strip()

PREFERRED_LLM_ENDPOINT = "databricks-gpt-5-6-sol"

if endpoint_names:
    if env_llm and env_llm in endpoint_names:
        default_llm = env_llm

    elif PREFERRED_LLM_ENDPOINT in endpoint_names:
        default_llm = PREFERRED_LLM_ENDPOINT

    else:
        default_llm = endpoint_names[0]

else:
    default_llm = env_llm

assert SQL_WAREHOUSE_ID, (
    "BLOCKED: SQL warehouse ID is empty."
)

assert LLM_ENDPOINT, (
    "BLOCKED: LLM endpoint is empty."
)

assert LLM_ENDPOINT == PREFERRED_LLM_ENDPOINT, (
    f"BLOCKED: wrong LLM endpoint selected: {LLM_ENDPOINT}. "
    f"Expected: {PREFERRED_LLM_ENDPOINT}"
)

print(
    "PASS Gate 11.1A — Databricks resources discovered"
)
# ------------------------------------------------------------
# CREATE / REFRESH WIDGETS
# ------------------------------------------------------------

for widget_name in [
    "stage11_sql_warehouse_id",
    "stage11_llm_endpoint",
]:
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass


dbutils.widgets.dropdown(
    "stage11_sql_warehouse_id",
    default_warehouse_id,
    all_warehouse_ids,
    "Stage 11 SQL warehouse ID",
)


if endpoint_names:
    dbutils.widgets.dropdown(
        "stage11_llm_endpoint",
        default_llm,
        endpoint_names,
        "Stage 11 LLM endpoint",
    )

else:
    dbutils.widgets.text(
        "stage11_llm_endpoint",
        default_llm,
        "Stage 11 LLM endpoint",
    )


# ------------------------------------------------------------
# READ SELECTED VALUES
# ------------------------------------------------------------

SQL_WAREHOUSE_ID = (
    dbutils.widgets
    .get("stage11_sql_warehouse_id")
    .strip()
)

LLM_ENDPOINT = (
    dbutils.widgets
    .get("stage11_llm_endpoint")
    .strip()
)


print(
    "\nSelected SQL warehouse ID:",
    SQL_WAREHOUSE_ID,
)

print(
    "Selected LLM endpoint     :",
    LLM_ENDPOINT or "(not selected yet)",
)

print()

assert SQL_WAREHOUSE_ID, (
    "BLOCKED: SQL warehouse ID is empty."
)

assert LLM_ENDPOINT, (
    "BLOCKED: LLM endpoint is empty."
)

print(
    "PASS Gate 11.1A — Databricks resources discovered"
)

### Lakebase connection

This reuses the **same SHIPP Lakebase Autoscaling target already used by Stage 12 validation**.

Credentials are generated dynamically by Databricks. No password or token is stored in the notebook.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.1B
# CONNECT TO REAL SHIPP LAKEBASE
# READ ONLY
# ============================================================

import psycopg

ENDPOINT_NAME = (
    "projects/shipp-capstone-v1/"
    "branches/production/endpoints/primary"
)

PGHOST = (
    "ep-dry-flower-d17p00tu."
    "database.us-west-2.cloud.databricks.com"
)

PGDATABASE = "databricks_postgres"
LAKEBASE_SCHEMA = "shipp"
PGUSER = w.current_user.me().user_name

def connect():
    credential = w.postgres.generate_database_credential(
        endpoint=ENDPOINT_NAME
    )

    return psycopg.connect(
        host=PGHOST,
        port=5432,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
        connect_timeout=10,
    )

with connect() as conn, conn.cursor() as cur:
    cur.execute(
        '''
        SELECT
            current_user,
            current_database(),
            to_regclass('shipp.listings'),
            to_regclass('shipp.requests'),
            to_regclass('shipp.saved_items'),
            to_regclass('shipp.agent_activity')
        '''
    )
    row = cur.fetchone()

print("Lakebase user       :", row[0])
print("Lakebase database   :", row[1])
print("shipp.listings      :", row[2])
print("shipp.requests      :", row[3])
print("shipp.saved_items   :", row[4])
print("shipp.agent_activity:", row[5])

assert row[1] == PGDATABASE
assert row[2] is not None
assert row[3] is not None
assert row[4] is not None
assert row[5] is not None

lakebase = LakebaseRepo(connect, LAKEBASE_SCHEMA)

print()
print("PASS Gate 11.1B — connected to real SHIPP Lakebase")

## Gate 11.2 — Choose one real cross-source validation fixture

The fixture must already exist in **Gold** and in **Lakebase**.

The notebook prefers a candidate whose request is still usable and whose listing is currently `AVAILABLE`.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.2
# FIND A REAL GOLD + LAKEBASE FIXTURE
# READ ONLY
# ============================================================

assert table_exists(GOLD_CANDIDATE_MATCHES), (
    f"BLOCKED: missing {GOLD_CANDIDATE_MATCHES}"
)

gold_df = spark.table(GOLD_CANDIDATE_MATCHES)

required_gold_columns = {
    "request_id",
    "listing_id",
    "match_score",
}

missing = sorted(required_gold_columns - set(gold_df.columns))
assert not missing, f"BLOCKED: Gold is missing columns: {missing}"

candidate_rows = (
    gold_df
    .select("request_id", "listing_id", "match_score")
    .where(
        F.col("request_id").isNotNull()
        & F.col("listing_id").isNotNull()
        & F.col("match_score").isNotNull()
    )
    .orderBy(F.col("match_score").desc(), F.col("request_id"), F.col("listing_id"))
    .collect()
)

assert candidate_rows, "BLOCKED: Gold has no candidate matches."

fixture = None

with connect() as conn, conn.cursor() as cur:
    for candidate in candidate_rows:
        cur.execute(
            f'''
            SELECT request_id, requester_id, status, request_text
            FROM {LAKEBASE_SCHEMA}.requests
            WHERE request_id = %s
            ''',
            (candidate["request_id"],),
        )
        req = cur.fetchone()

        cur.execute(
            f'''
            SELECT listing_id, status, available_until, title
            FROM {LAKEBASE_SCHEMA}.listings
            WHERE listing_id = %s
            ''',
            (candidate["listing_id"],),
        )
        lst = cur.fetchone()

        if not req or not lst:
            continue

        request_status = str(req[2] or "").upper()
        listing_status = str(lst[1] or "").upper()

        if request_status != "CLOSED" and listing_status == "AVAILABLE":
            fixture = {
                "request_id": str(req[0]),
                "user_id": str(req[1]),
                "request_status": str(req[2]),
                "request_text": str(req[3] or ""),
                "listing_id": str(lst[0]),
                "listing_status": str(lst[1]),
                "available_until": lst[2],
                "listing_title": str(lst[3] or ""),
                "gold_match_score": float(candidate["match_score"]),
            }
            break

assert fixture is not None, (
    "BLOCKED: no Gold candidate is currently backed by a usable request "
    "and AVAILABLE Lakebase listing."
)

TEST_REQUEST_ID = fixture["request_id"]
TEST_USER_ID = fixture["user_id"]
TEST_LISTING_ID = fixture["listing_id"]
TEST_QUERY = (
    fixture["request_text"].strip()
    or fixture["listing_title"].strip()
    or "household item that fits this request"
)

print("Selected Stage 11 fixture:")
for key, value in fixture.items():
    print(f"  {key}: {value}")

print()
print("PASS Gate 11.2 — real Gold + Lakebase fixture selected")

## Gate 11.3 — `get_candidate_matches()` against real Gold

This is the existing `GoldReader` implementation used by the Agent and the Databricks App.  
It uses the Databricks SQL Statement Execution API — no Spark dependency inside the Agent.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.3
# REAL GOLD READER
# READ ONLY
# ============================================================

SQL_WAREHOUSE_ID = dbutils.widgets.get("stage11_sql_warehouse_id").strip()
assert SQL_WAREHOUSE_ID, (
    "BLOCKED: choose a value in the Stage 11 SQL warehouse widget."
)

gold_reader = GoldReader(
    w,
    SQL_WAREHOUSE_ID,
    GOLD_CANDIDATE_MATCHES,
)

matches = gold_reader.get_candidate_matches(
    TEST_REQUEST_ID,
    limit=5,
    min_score=0.0,
)

print(f"Gold matches returned for {TEST_REQUEST_ID}: {len(matches)}")

for match in matches:
    print(match)

assert matches, "FAIL: get_candidate_matches returned no matches for the known request."
assert all(m.request_id == TEST_REQUEST_ID for m in matches)
assert all(m.listing_id for m in matches)
assert all(0.0 <= m.match_score <= 1.0 for m in matches)
assert TEST_LISTING_ID in {m.listing_id for m in matches}, (
    f"FAIL: fixture listing {TEST_LISTING_ID} was not returned by GoldReader."
)

CANDIDATE_IDS = [m.listing_id for m in matches]

print()
print("Candidate IDs:", CANDIDATE_IDS)
print("PASS Gate 11.3 — get_candidate_matches() returned real trusted Gold candidates")

## Gate 11.4 — `search_listing_context()` against real AI Search

Search must be **scoped to the Gold candidate IDs**.  
A search result outside `CANDIDATE_IDS` is a failure.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.4
# REAL AI SEARCH
# READ ONLY
# ============================================================

listing_search = ListingSearch(
    w,
    VS_INDEX_NAME,
)

search_hits = listing_search.search(
    TEST_QUERY,
    listing_ids=CANDIDATE_IDS,
    k=min(5, len(CANDIDATE_IDS)),
)

print("Search query:", TEST_QUERY)
print("Allowed Gold candidate IDs:", CANDIDATE_IDS)
print(f"AI Search hits: {len(search_hits)}")

for hit in search_hits:
    print(hit)

assert search_hits, (
    "FAIL: AI Search returned no context for the real Gold candidate set."
)

outside_gold = [
    hit.listing_id
    for hit in search_hits
    if hit.listing_id not in CANDIDATE_IDS
]

assert not outside_gold, (
    f"FAIL: AI Search returned listings outside Gold candidates: {outside_gold}"
)

TRACE_LISTING_ID = search_hits[0].listing_id

print()
print("Selected cross-source listing_id:", TRACE_LISTING_ID)
print("PASS Gate 11.4 — AI Search returned candidate-scoped listing context")

## Gate 11.5 — `get_listing_status()` from real Lakebase

Lakebase is the operational source of truth for the listing's current state.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.5
# REAL LAKEBASE CURRENT STATUS
# READ ONLY
# ============================================================

listing_state = lakebase.get_listing(TRACE_LISTING_ID)

print("Listing ID        :", TRACE_LISTING_ID)
print("Lakebase state    :", listing_state)

assert listing_state is not None, (
    f"FAIL: {TRACE_LISTING_ID} exists in Gold/Search but not in Lakebase."
)

assert listing_state.listing_id == TRACE_LISTING_ID

print()
print("Cross-source trace:")
print("  Gold      :", TRACE_LISTING_ID in CANDIDATE_IDS)
print("  AI Search :", TRACE_LISTING_ID in {h.listing_id for h in search_hits})
print("  Lakebase  :", listing_state.listing_id == TRACE_LISTING_ID)

assert TRACE_LISTING_ID in CANDIDATE_IDS
assert TRACE_LISTING_ID in {h.listing_id for h in search_hits}

print()
print("PASS Gate 11.5 — same listing_id traced through Gold → AI Search → Lakebase")

## Gate 11.6 — Validate the actual Agent READ tool layer

This uses the existing `ToolBox` handlers:

- `get_candidate_matches`
- `search_listing_context`
- `get_listing_status`

**Note:** `ToolBox.dispatch()` writes audit events to `shipp.agent_activity`.  
It does **not** write `saved_items`.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.6A
# BUILD EXISTING TOOLBOX
# ============================================================

LLM_ENDPOINT = dbutils.widgets.get("stage11_llm_endpoint").strip()

settings = AgentSettings(
    llm_endpoint=LLM_ENDPOINT or "stage11-validation-placeholder",
    sql_warehouse_id=SQL_WAREHOUSE_ID,
    candidate_matches_table=GOLD_CANDIDATE_MATCHES,
    search_index=VS_INDEX_NAME,
    lakebase_schema=LAKEBASE_SCHEMA,
    lakebase_instance=None,
    max_matches=5,
    max_search_results=5,
    max_tool_rounds=6,
    min_match_score=0.0,
)

toolbox = ToolBox(
    ToolContext(
        user_id=TEST_USER_ID,
        request_id=TEST_REQUEST_ID,
    ),
    settings,
    gold_reader,
    listing_search,
    lakebase,
)

with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f'''
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        WHERE user_id = %s
          AND request_id = %s
        ''',
        (TEST_USER_ID, TEST_REQUEST_ID),
    )
    SAVED_ITEMS_BEFORE_READ_TEST = cur.fetchone()[0]

print("Saved items before Agent READ tool validation:", SAVED_ITEMS_BEFORE_READ_TEST)
print("PASS Gate 11.6A — ToolBox constructed")

In [0]:
# ============================================================
# STAGE 11 — GATE 11.6B
# CALL ALL THREE AGENT READ TOOLS
# ============================================================

candidate_result = toolbox.dispatch(
    "get_candidate_matches",
    "{}",
)

search_result = toolbox.dispatch(
    "search_listing_context",
    json.dumps({"query": TEST_QUERY}),
)

status_result = toolbox.dispatch(
    "get_listing_status",
    json.dumps({"listing_id": TRACE_LISTING_ID}),
)

print("get_candidate_matches:")
print(json.dumps(candidate_result, indent=2, default=str))

print("\nsearch_listing_context:")
print(json.dumps(search_result, indent=2, default=str))

print("\nget_listing_status:")
print(json.dumps(status_result, indent=2, default=str))

assert candidate_result.get("matches"), (
    "FAIL: Agent tool get_candidate_matches returned no matches."
)

assert search_result.get("results"), (
    "FAIL: Agent tool search_listing_context returned no results."
)

assert status_result.get("listing_id") == TRACE_LISTING_ID, (
    "FAIL: get_listing_status did not return the requested candidate."
)

assert "error" not in candidate_result
assert "error" not in search_result
assert "error" not in status_result

print()
print("PASS Gate 11.6B — all three Agent READ tools succeeded on real sources")

## Gate 11.7 — Safe failure / no-hallucination cases

P0 negative behavior:

1. unknown request → no Gold matches
2. empty candidate scope → no AI Search results
3. fabricated/non-candidate listing ID → rejected by Agent tool guardrail
4. unknown listing → Lakebase returns `None`

In [0]:
# ============================================================
# STAGE 11 — GATE 11.7
# NEGATIVE / SAFE FAILURE TESTS
# ============================================================

UNKNOWN_REQUEST_ID = "stage11-validation-unknown-request"
UNKNOWN_LISTING_ID = "stage11-validation-unknown-listing"

unknown_matches = gold_reader.get_candidate_matches(
    UNKNOWN_REQUEST_ID,
    limit=5,
    min_score=0.0,
)

empty_search = listing_search.search(
    "anything",
    listing_ids=[],
    k=3,
)

rejected_status = toolbox.dispatch(
    "get_listing_status",
    json.dumps({"listing_id": UNKNOWN_LISTING_ID}),
)

missing_listing = lakebase.get_listing(UNKNOWN_LISTING_ID)

print("Unknown request Gold matches :", unknown_matches)
print("Empty-scope AI Search        :", empty_search)
print("Non-candidate tool result     :", rejected_status)
print("Unknown Lakebase listing      :", missing_listing)

assert unknown_matches == [], (
    "FAIL: unknown request unexpectedly returned Gold candidates."
)

assert empty_search == [], (
    "FAIL: AI Search should return [] when no candidate IDs are allowed."
)

assert "error" in rejected_status, (
    "FAIL: fabricated/non-candidate listing_id was not rejected."
)

assert missing_listing is None, (
    "FAIL: unknown listing should not resolve in Lakebase."
)

print()
print("PASS Gate 11.7 — no-result and fabricated-ID cases fail safely")

## Gate 11.8 — Live grounded Agent turn

This is the final Stage 11 runtime proof.

Before running this cell, confirm the notebook widget **`stage11_llm_endpoint`** is set to the LLM endpoint you want SHIPP to use.

The prompt explicitly requires all three READ tools and forbids a save proposal.

In [0]:
# ============================================================
# STAGE 11 — GATE 11.8A
# FIND AN LLM ENDPOINT COMPATIBLE WITH THE EXISTING AGENT
# READ ONLY
# ============================================================

print("=== STAGE 11.8A — LLM ENDPOINT COMPATIBILITY CHECK ===")

client = w.serving_endpoints.get_open_ai_client()

endpoint_names = [
    getattr(ep, "name", None)
    for ep in w.serving_endpoints.list()
    if getattr(ep, "name", None)
]

print("Visible endpoints:")
for name in endpoint_names:
    print("  ", name)

assert endpoint_names, "BLOCKED: no serving endpoints are visible."

probe_tools = [
    {
        "type": "function",
        "function": {
            "name": "stage11_probe",
            "description": "Temporary compatibility probe.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    }
]

compatible = []

for name in endpoint_names:
    try:
        response = client.chat.completions.create(
            model=name,
            messages=[
                {
                    "role": "user",
                    "content": "Reply only with OK. Do not call a tool.",
                }
            ],
            tools=probe_tools,
            tool_choice="auto",
            temperature=0.0,
        )

        message = response.choices[0].message

        print(f"\nPASS: {name}")
        print("  chat.completions supported")
        print("  tools parameter supported")
        print("  response:", message.content)

        compatible.append(name)
        break

    except Exception as exc:
        print(
            f"\nSKIP: {name}\n"
            f"  {type(exc).__name__}: {str(exc)[:250]}"
        )


assert compatible, (
    "BLOCKED: none of the visible endpoints supports the "
    "chat.completions + tools interface required by the existing SHIPP Agent."
)

COMPATIBLE_LLM_ENDPOINT = compatible[0]

print("\n==============================================")
print("USE THIS ENDPOINT:", COMPATIBLE_LLM_ENDPOINT)
print("==============================================")

# Recreate the existing notebook widget with the compatible endpoint selected.
try:
    dbutils.widgets.remove("stage11_llm_endpoint")
except Exception:
    pass

dbutils.widgets.dropdown(
    "stage11_llm_endpoint",
    COMPATIBLE_LLM_ENDPOINT,
    compatible,
    "Stage 11 LLM endpoint",
)

print("PASS Gate 11.8A — compatible Agent LLM endpoint selected")

In [0]:
# ============================================================
# STAGE 11 — GATE 11.8
# LIVE AGENT READ FLOW
# ============================================================

LLM_ENDPOINT = dbutils.widgets.get("stage11_llm_endpoint").strip()

assert LLM_ENDPOINT, (
    "BLOCKED: select the SHIPP LLM endpoint in the stage11_llm_endpoint widget."
)

settings_live = AgentSettings(
    llm_endpoint=LLM_ENDPOINT,
    sql_warehouse_id=SQL_WAREHOUSE_ID,
    candidate_matches_table=GOLD_CANDIDATE_MATCHES,
    search_index=VS_INDEX_NAME,
    lakebase_schema=LAKEBASE_SCHEMA,
    lakebase_instance=None,
    max_matches=5,
    max_search_results=5,
    max_tool_rounds=6,
    min_match_score=0.0,
)

model = DatabricksChatModel(
    LLM_ENDPOINT,
    w,
)

agent = ShippAgent(
    settings=settings_live,
    model=model,
    gold=gold_reader,
    search=listing_search,
    lakebase=lakebase,
)

user_message = (
    "Use get_candidate_matches first. Then use search_listing_context to inspect "
    "supporting description or image-derived context for the best candidate. "
    "Then use get_listing_status to verify its current Lakebase state. "
    "Give me a grounded recommendation based only on those tool results. "
    "Do not call propose_save and do not save anything."
)

turn = agent.chat(
    user_id=TEST_USER_ID,
    request_id=TEST_REQUEST_ID,
    user_message=user_message,
)

print("Agent reply:")
print(turn.reply)

print("\nTool trace:")
for step in turn.tool_trace:
    print(json.dumps(step, indent=2, default=str))

print("\nPending save:", turn.pending_save)

tools_used = [step["tool"] for step in turn.tool_trace]

required_tools = {
    "get_candidate_matches",
    "search_listing_context",
    "get_listing_status",
}

missing_tools = sorted(required_tools - set(tools_used))

assert turn.reply.strip(), "FAIL: Agent returned an empty reply."
assert not missing_tools, (
    f"FAIL: live Agent did not use all required READ tools. Missing: {missing_tools}"
)

assert "propose_save" not in tools_used, (
    "FAIL: Stage 11 validation asked for READ only, but Agent called propose_save."
)

for step in turn.tool_trace:
    assert "error" not in step["result"], (
        f"FAIL: tool {step['tool']} returned an error: {step['result']}"
    )

assert turn.pending_save is None, (
    "FAIL: Stage 11 READ validation must not create a pending save."
)

print()
print("Tools used:", tools_used)
print("PASS Gate 11.8 — live grounded Agent READ flow used Gold + AI Search + Lakebase")

## Final Stage 11 closure

The final gate proves the READ path did not create a saved-item business action.

Audit rows in `agent_activity` are allowed and expected.

In [0]:
# ============================================================
# STAGE 11 — FINAL VALIDATION
# ============================================================

with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f'''
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        WHERE user_id = %s
          AND request_id = %s
        ''',
        (TEST_USER_ID, TEST_REQUEST_ID),
    )
    saved_items_after = cur.fetchone()[0]

    cur.execute(
        f'''
        SELECT tool_name, action_status, entity_id, created_at
        FROM {LAKEBASE_SCHEMA}.agent_activity
        WHERE user_id = %s
        ORDER BY created_at DESC
        LIMIT 20
        ''',
        (TEST_USER_ID,),
    )
    recent_activity = cur.fetchall()

print("Saved items before Stage 11 READ:", SAVED_ITEMS_BEFORE_READ_TEST)
print("Saved items after Stage 11 READ :", saved_items_after)

print("\nRecent Agent activity:")
for row in recent_activity:
    print(row)

assert saved_items_after == SAVED_ITEMS_BEFORE_READ_TEST, (
    "FAIL: Stage 11 READ changed saved_items."
)

print()
print("==============================================")
print("STAGE 11 — AGENT READ")
print("IMPLEMENTED : PASS")
print("REAL GOLD   : PASS")
print("AI SEARCH   : PASS")
print("LAKEBASE    : PASS")
print("TRACEABILITY: PASS")
print("NEGATIVE    : PASS")
print("LIVE AGENT  : PASS")
print("NO SAVE     : PASS")
print("----------------------------------------------")
print("FINAL STATUS: VALIDATED")
print("==============================================")
print()
print("PASS Stage 11 — Agent READ VALIDATED")